# FSR v2 Dev Ingest — Direct Job Submission Test
Run inside a Databricks cluster. No external imports needed.

In [ ]:
# ── Edit these before running ──────────────────────────────────────────────────
# P1 expects PDF filenames (not document IDs) as they appear in the volume
TARGET_PDF_NAMES = (
    "35803273-2440-4d36-87fe-e45f7f0e5467_605011422-40815-270T483-Final_Master_Report.pdf,"
    "806975e9-062b-4da3-9ae7-5881a5daffd1_204049598-39387-297652-Final_Master_Report.pdf"
)
TABLE_PROFILE    = "ms_test"   # ms_test | ds_test | dev
MODE             = "no-drop"   # no-drop keeps recreated tables; use drop only to reset

ORCHESTRATOR_NOTEBOOK = "/Workspace/Users/madhurima.saxena@gevernova.com/pw_sdg_ai_ser_repo/validation/fsr_v2/ingest_data/nb_fsr_v2_dev_ingest"
# ───────────────────────────────────────────────────────────────────────────────

PROFILE_CONFIGS = {
    "dev": {
        "metadata_table":         "vaid.ai_sot_field_service_report.fsr_metadata_v2",
        "chunk_table":            "vaid.ai_std_con_field_service_report.fsr_chunks_v2",
        "doc_equipment_map_table":"vaid.ai_sot_field_service_report.fsr_document_equipment_map_v2",
        "run_log_table":          "vaid.ai_sot_field_service_report.fsr_run_log_v2",
        "dq_log_table":           "vaid.ai_sot_field_service_report.fsr_data_quality_log_v2",
        "vs_index":               "vaid.ai_std_con_field_service_report.fsr_vs_index_v2",
        "vs_endpoint":            "pw-ser-sdg-vector-search",
    },
    "ds_test": {
        "metadata_table":         "vaid.ai_sot_field_service_report.ds_test_fsr_metadata_v2",
        "chunk_table":            "vaid.ai_std_con_field_service_report.ds_test_fsr_chunks_v2",
        "doc_equipment_map_table":"vaid.ai_sot_field_service_report.ds_test_fsr_document_equipment_map_v2",
        "run_log_table":          "vaid.ai_sot_field_service_report.ds_test_fsr_run_log_v2",
        "dq_log_table":           "vaid.ai_sot_field_service_report.ds_test_fsr_data_quality_log_v2",
        "vs_index":               "vaid.ai_std_con_field_service_report.ds_test_fsr_vs_index_v2",
        "vs_endpoint":            "pw-ser-sdg-vector-search",
    },
    "ms_test": {
        "metadata_table":         "vaid.ai_sot_field_service_report.ms_test_fsr_metadata_v2",
        "chunk_table":            "vaid.ai_std_con_field_service_report.ms_test_fsr_chunks_v2",
        "doc_equipment_map_table":"vaid.ai_sot_field_service_report.ms_test_fsr_document_equipment_map_v2",
        "run_log_table":          "vaid.ai_sot_field_service_report.ms_test_fsr_run_log_v2",
        "dq_log_table":           "vaid.ai_sot_field_service_report.ms_test_fsr_data_quality_log_v2",
        "vs_index":               "vaid.ai_std_con_field_service_report.ms_test_fsr_vs_index_v2",
        "vs_endpoint":            "pw-ser-sdg-vector-search",
    },
}

cfg = PROFILE_CONFIGS[TABLE_PROFILE]
print(f"Profile    : {TABLE_PROFILE}")
print(f"Mode       : {MODE}")
print(f"Docs       : {TARGET_PDF_NAMES}")
print(f"Notebook   : {ORCHESTRATOR_NOTEBOOK}")
print(f"Metadata → : {cfg['metadata_table']}")

In [ ]:
notebook_params = {
    "jb_env":                    "dev",
    "TARGET_PDF_NAMES":          TARGET_PDF_NAMES,
    "METADATA_TABLE_V2":         cfg["metadata_table"],
    "CHUNK_TABLE_V2":            cfg["chunk_table"],
    "DOC_EQUIPMENT_MAP_TABLE_V2":cfg["doc_equipment_map_table"],
    "RUN_LOG_TABLE_V2":          cfg["run_log_table"],
    "DQ_LOG_TABLE_V2":           cfg["dq_log_table"],
    "VS_INDEX_V2":               cfg["vs_index"],
    "VS_ENDPOINT_V2":            cfg["vs_endpoint"],
    "DROP_TABLES_AND_INDEX":     "true" if MODE == "drop" else "false",
    "RUN_ALL_FROM_VOLUMES":      "false",
    "FSR_CHUNKING_STRATEGY":     "section",
    "FSR_SOURCE_VOLUME_PATHS":   (
        "/Volumes/viud/ing_ud_fieldvision/fv_field_service_report,"
        "/Volumes/viud/ing_ud_fsr_manual/manual_field_service_report/ecrt_reports,"
        "/Volumes/viud/ing_ud_fsr_manual/manual_field_service_report/FSR_manual"
    ),
    "LITELLM_BASE_URL":          "https://dev-gateway.apps.gevernova.net",
    "FSR_PARSED_DOC_VOLUME_ROOT": "/Volumes/vaid/ai_sot_field_service_report/fsr_parsed_docs",
}

print(f"Profile : {TABLE_PROFILE} | Mode : {MODE}")
print(f"Docs    : {TARGET_PDF_NAMES}")
print(f"Notebook: {ORCHESTRATOR_NOTEBOOK}")
print("\nRunning orchestrator (blocks until pipeline completes)...\n")

result = dbutils.notebook.run(ORCHESTRATOR_NOTEBOOK, timeout_seconds=3600, arguments=notebook_params)

print(f"\n✅ Done! Result: {result}")

## Deployed Agent API Test (OAuth token only)
Use a single auth method to avoid retries: set `APP_OAUTH_TOKEN` widget with OAuth access token from profile `dev-oauth`, then run the next cell.

In [ ]:
# Paste OAuth token here before running the API test cell.
dbutils.widgets.text("APP_OAUTH_TOKEN", "")
print("Widget ready: APP_OAUTH_TOKEN")
print("Paste the OAuth access_token value into this widget, then run the next test cell.")

In [ ]:
# No package install required for the current API test flow.
# If you ran old setup cells, you can ignore this cell.
print("No setup needed. Proceed to the next cell.")

In [ ]:
import json
import requests

APP_NAME = "agent-fsr-v2-dev-ingest"
APP_URL = "https://agent-fsr-v2-dev-ingest-7474648066331722.aws.databricksapps.com"
USER_PROMPT = "Ingest docs 4597a853-5ffb-40bb-b0be-bd6307b0acdc,796f4d53-a8ad-42e1-af4d-53a8add2e1a4, using ms_test with no-drop"

# Token must come from widget only (safer than hardcoding).
dbutils.widgets.text("APP_OAUTH_TOKEN", "")
oauth_token = dbutils.widgets.get("APP_OAUTH_TOKEN").strip()

# Guard against pasting with Bearer prefix.
if oauth_token.lower().startswith("bearer "):
    oauth_token = oauth_token[7:].strip()

if not oauth_token:
    raise RuntimeError(
        "APP_OAUTH_TOKEN is empty. Put raw OAuth access_token in the widget (no Bearer prefix)."
    )

if oauth_token.startswith("dapi") or oauth_token.startswith("dapix"):
    raise RuntimeError(
        "Detected PAT token (starts with dapi). Use OAuth access token from profile dev-oauth."
    )

url = f"{APP_URL.rstrip('/')}/responses"
headers = {
    "Authorization": f"Bearer {oauth_token}",
    "Content-Type": "application/json",
}
payload = {
    "input": [{"role": "user", "content": USER_PROMPT}],
    "stream": False,
}

print("Testing deployed app endpoint:")
print(f"  app      : {APP_NAME}")
print(f"  endpoint : {url}")
print("  token    : widget:APP_OAUTH_TOKEN")
print(f"  tokenLen : {len(oauth_token)}")

resp = requests.post(url, headers=headers, json=payload, timeout=180, allow_redirects=True)
content_type = (resp.headers.get("content-type") or "").lower()
body = resp.text or ""

print(f"\nHTTP_STATUS:{resp.status_code}")
print(f"CONTENT_TYPE:{content_type}")

if "text/html" in content_type and "Databricks - Sign In" in body:
    raise RuntimeError(
        "Token not accepted for Apps API (received Sign-In HTML). Regenerate OAuth token from dev-oauth and retry."
    )

if content_type.startswith("application/json"):
    try:
        print("\nResponse body (JSON):\n")
        print(json.dumps(resp.json(), indent=2)[:8000])
    except Exception:
        print("\nResponse body (text):\n")
        print(body[:8000])
else:
    print("\nResponse body (text):\n")
    print(body[:8000])

resp.raise_for_status()

print("\nEquivalent curl command (redacted token):\n")
print(
    "curl --request POST \\\n"
    f"  --url {url} \\\n"
    "  --header 'Authorization: Bearer <OAUTH_TOKEN>' \\\n"
    "  --header 'Content-Type: application/json' \\\n"
    f"  --data '{json.dumps(payload)}'"
    )